In [5]:
%pip install pysindy

In [6]:
"""
On the Role of Candidate Library Design in Sparse Identification 
of Nonlinear Dynamical Systems

AMATH 445: Scientific Machine Learning
University of Waterloo, April 2026
"""

import pysindy as ps
import numpy as np
from scipy.integrate import solve_ivp
from itertools import combinations_with_replacement
import matplotlib.pyplot as plt
import time

# Lorenz — Data Generation

def lorenz(t, state, sigma=10.0, beta=8.0/3.0, rho=28.0):
    x, y, z = state
    dxdt = sigma * (y - x)
    dydt = x * (rho - z) - y
    dzdt = x * y - beta * z
    return [dxdt, dydt, dzdt]

# Parameters sigma=10, beta=8/3, rho=28
x0 = [-8.0, 7.0, 27.0]
t_span = (0, 50)
dt = 0.001
t_eval = np.arange(t_span[0], t_span[1], dt)

sol = solve_ivp(lorenz, t_span, x0, method='RK45',
                t_eval=t_eval, rtol=1e-10, atol=1e-10)

X = sol.y.T 
t = sol.t

# derivatives
Xdot = np.zeros_like(X)
for i in range(len(t)):
    Xdot[i] = lorenz(t[i], X[i])

print(f"Data generated: {X.shape[0]} time steps, {X.shape[1]} variables")

# Library

def build_library(X, degree):
    m, n = X.shape
    Theta = np.ones((m, 1))
    labels = ['1']
    var_names = ['x', 'y', 'z'][:n]
    
    for d in range(1, degree + 1):
        for combo in combinations_with_replacement(range(n), d):
            col = np.ones(m)
            name = ''
            for idx in combo:
                col *= X[:, idx]
                name += var_names[idx]
            Theta = np.column_stack([Theta, col])
            labels.append(name)
    
    return Theta, labels

# STLS
def stls(Theta, xdot_col, threshold, max_iter=100):
    xi, _, _, _ = np.linalg.lstsq(Theta, xdot_col, rcond=None)
    
    for _ in range(max_iter):
        small = np.abs(xi) < threshold
        xi[small] = 0.0
        active = ~small
        if np.sum(active) == 0:
            break
        xi_active, _, _, _ = np.linalg.lstsq(
            Theta[:, active], xdot_col, rcond=None
        )
        xi[active] = xi_active
        if np.array_equal(small, np.abs(xi) < threshold):
            break
    
    return xi


def run_sindy(Theta, Xdot, threshold=0.1):
    n_eq = Xdot.shape[1]
    p = Theta.shape[1]
    Xi = np.zeros((p, n_eq))
    for i in range(n_eq):
        Xi[:, i] = stls(Theta, Xdot[:, i], threshold)
    return Xi

# Noise

'''
noise is added to the state measurements and then the derivatives are computed.
'''
def add_noise(X_clean, noise_level):
    if noise_level == 0:
        return X_clean.copy()
    return X_clean + noise_level * np.random.randn(*X_clean.shape)


def finite_diff(X, dt):
    Xdot = np.zeros_like(X)
    Xdot[0] = (X[1] - X[0]) / dt
    Xdot[1:-1] = (X[2:] - X[:-2]) / (2 * dt)
    Xdot[-1] = (X[-1] - X[-2]) / dt
    return Xdot

# Metrics

def get_true_coefficients(labels):
    p = len(labels)
    Xi_true = np.zeros((p, 3))
    for i, label in enumerate(labels):
        if label == 'x':   Xi_true[i, 0] = -10.0
        if label == 'y':   Xi_true[i, 0] = 10.0
        if label == 'x':   Xi_true[i, 1] = 28.0
        if label == 'y':   Xi_true[i, 1] = -1.0
        if label == 'xz':  Xi_true[i, 1] = -1.0
        if label == 'z':   Xi_true[i, 2] = -8.0/3.0
        if label == 'xy':  Xi_true[i, 2] = 1.0
    return Xi_true


def check_support(Xi_true, Xi_recovered, tol=1e-6):
    true_nz = np.abs(Xi_true) > tol
    rec_nz = np.abs(Xi_recovered) > tol
    return np.array_equal(true_nz, rec_nz)


def coeff_error(Xi_true, Xi_recovered):
    return np.linalg.norm(Xi_true - Xi_recovered) / np.linalg.norm(Xi_true)

# Degree 2, no noise

print("\n")
print("Baseline, SINDy on Lorenz, degree 2, no noise")

Theta2, labels2 = build_library(X, 2)
Xi2 = run_sindy(Theta2, Xdot, threshold=0.1)

var_names = ['dx/dt', 'dy/dt', 'dz/dt']
for j in range(3):
    terms = []
    for i, label in enumerate(labels2):
        if abs(Xi2[i, j]) > 1e-10:
            terms.append(f"{Xi2[i,j]:+.4f}*{label}")
    print(f"  {var_names[j]} = {' '.join(terms)}")

print("\nTrue Lorenz equations:")
print("  dx/dt = -10*x + 10*y")
print("  dy/dt = +28*x - 1*y - 1*xz")
print("  dz/dt = -2.6667*z + 1*xy")

# PySINDy Check

print("\n")
print("PySINDy")

model = ps.SINDy(
    feature_library=ps.PolynomialLibrary(degree=2),
    optimizer=ps.STLSQ(threshold=0.1)
)
model.fit(X, t=dt)
model.print()

# Library Degree x Noise 


print("\n")
print("Library experiement")

degrees = [1, 2, 3, 4, 5]
noise_levels = [0, 0.01, 0.1, 1.0, 5.0]
threshold = 0.1

results = []

for deg in degrees:
    for noise in noise_levels:
        np.random.seed(42)
        X_noisy = add_noise(X, noise)
        
        if noise == 0:
            Xdot_used = Xdot
        else:
            Xdot_used = finite_diff(X_noisy, dt)
        
        Theta_lib, labels_lib = build_library(X_noisy, deg)
        Xi_rec = run_sindy(Theta_lib, Xdot_used, threshold)
        Xi_true = get_true_coefficients(labels_lib)
        
        support_ok = check_support(Xi_true, Xi_rec)
        c_error = coeff_error(Xi_true, Xi_rec)
        
        results.append({
            'degree': deg,
            'noise': noise,
            'library_size': Theta_lib.shape[1],
            'support_correct': support_ok,
            'coeff_error': c_error
        })

# Print summary table
print(f"\n{'Deg':<5} {'Noise':<8} {'Terms':<7} {'Coeff Err':<12} {'Support'}")
print("-" * 50)
for r in results:
    mark = "true" if r['support_correct'] else "false"
    print(f"{r['degree']:<5} {r['noise']:<8.2f} {r['library_size']:<7} "
          f"{r['coeff_error']:<12.6f} {mark}")

# Degree 5 seed analysis

print("\n")
print("Seed analysis: Degree 5, Noise=0.01")

for seed in [0, 1, 2, 42, 99]:
    np.random.seed(seed)
    X_noisy = add_noise(X, 0.01)
    Xdot_noisy = finite_diff(X_noisy, dt)
    Theta5, labels5 = build_library(X_noisy, 5)
    Xi5 = run_sindy(Theta5, Xdot_noisy, threshold=0.1)
    Xi_true = get_true_coefficients(labels5)
    ok = check_support(Xi_true, Xi5)
    err = coeff_error(Xi_true, Xi5)
    print(f"  seed={seed}: support={'true' if ok else 'false'}, error={err:.4f}")

# Failing class analysis

print("\n")
print("Failing classes")

cases = [
    (5, 0.01, "Deg 5, noise=0.01 — fail"),
    (2, 1.00, "Deg 2, noise=1.0 — fail"),
]

for deg, noise, desc in cases:
    print(f"\n{desc}")
    np.random.seed(42)
    X_noisy = add_noise(X, noise)
    Xdot_noisy = finite_diff(X_noisy, dt) if noise > 0 else Xdot
    Theta_lib, labels_lib = build_library(X_noisy, deg)
    Xi_rec = run_sindy(Theta_lib, Xdot_noisy, threshold=0.1)
    
    for j in range(3):
        terms = []
        for i, label in enumerate(labels_lib):
            if abs(Xi_rec[i, j]) > 1e-10:
                terms.append(f"{Xi_rec[i,j]:+.4f}*{label}")
        print(f"  {var_names[j]} = {' '.join(terms)}")
        print(f"    ({len(terms)} active terms)")

# Computation time

print("\n")
print("COMPUTATION TIME COMPARISON")
print(f"{'Degree':<8} {'Terms':<8} {'Time (s)':<12} {'Support'}")

for deg in [1, 2, 3, 4, 5]:
    np.random.seed(42)
    start = time.time()
    Theta_lib, labels_lib = build_library(X, deg)
    Xi_rec = run_sindy(Theta_lib, Xdot, threshold=0.1)
    elapsed = time.time() - start
    Xi_true = get_true_coefficients(labels_lib)
    ok = check_support(Xi_true, Xi_rec)
    print(f"{deg:<8} {Theta_lib.shape[1]:<8} {elapsed:<12.4f} {'true' if ok else 'false'}")

# Threshold test with degree 2 noise 1.0

print("\n")
print("Threshold test: Degree 2, Noise=1.0")
print(f"{'λ':<8} {'Coeff Err':<12} {'Active Terms':<14} {'Support'}")

for lam in [0.05, 0.1, 0.2, 0.5, 1.0, 2.0]:
    np.random.seed(42)
    X_noisy = add_noise(X, 1.0)
    Xdot_noisy = finite_diff(X_noisy, dt)
    Theta_lib, labels_lib = build_library(X_noisy, 2)
    Xi_rec = run_sindy(Theta_lib, Xdot_noisy, threshold=lam)
    Xi_true = get_true_coefficients(labels_lib)
    ok = check_support(Xi_true, Xi_rec)
    err = coeff_error(Xi_true, Xi_rec)
    n_terms = np.sum(np.abs(Xi_rec) > 1e-10)
    print(f"{lam:<8} {err:<12.4f} {n_terms:<14} {'true' if ok else 'false'}")

Data generated: 50000 time steps, 3 variables


Baseline, SINDy on Lorenz, degree 2, no noise
  dx/dt = -10.0000*x +10.0000*y
  dy/dt = +28.0000*x -1.0000*y -1.0000*xz
  dz/dt = -2.6667*z +1.0000*xy

True Lorenz equations:
  dx/dt = -10*x + 10*y
  dy/dt = +28*x - 1*y - 1*xz
  dz/dt = -2.6667*z + 1*xy


PySINDy
(x0)' = -10.000 x0 +  10.000 x1
(x1)' =  27.998 x0 + -1.000 x1 + -1.000 x0 x2
(x2)' = -2.667 x2 +  1.000 x0 x1


Library experiement

Deg   Noise    Terms   Coeff Err    Support
--------------------------------------------------
1     0.00     4       1.356088     false
1     0.01     4       1.356060     false
1     0.10     4       1.355418     false
1     1.00     4       1.307096     false
1     5.00     4       1.238812     false
2     0.00     10      0.000000     true
2     0.01     10      0.000065     true
2     0.10     10      0.002088     true
2     1.00     10      0.546025     false
2     5.00     10      1.595240     false
3     0.00     20      0.000000     true
3